# 实验 1：认识 Qwen3-0.6B 的外部骨架

## 今天只回答一个问题

**一个 token 进入 Qwen3-0.6B 之后，要依次经过哪些主要模块？**

本实验只观察模型的**顶层流水线**和**模块树**：

1. 先看一张总览图，建立整体地图；
2. 再从真实模型对象自动生成一棵精简树；
3. 最后把图和树放在一起，确认它们描述的是同一副骨架。

今天暂时不打开 Transformer Block 的内部。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。模型会加载约 1.2GB 权重，但本实验不生成文本、不训练模型。

In [ ]:
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM


def find_project_root() -> Path:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'models' / 'Qwen3-0.6B-Base').is_dir():
            return directory
    raise FileNotFoundError('找不到 models/Qwen3-0.6B-Base')


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / 'models' / 'Qwen3-0.6B-Base'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'模型目录: {MODEL_PATH}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')


In [ ]:
# 这里只加载模型，不做生成；设备固定为 CPU，方便专注观察结构。
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
).cpu().eval()

print(f'模型类: {type(model).__name__}')
print(f'模型设备: {next(model.parameters()).device}')


## 1. 先看模型的总览图

今天只观察模型的**外部流水线**，先不打开 Transformer Block 的内部。

<div align="center">
  <img src="../assets/qwen3-backbone-overview.png" alt="Qwen3 顶层结构总览图" width="360" style="max-width: 100%; border: 1px solid #cbd5e1; border-radius: 8px;">
</div>

```text
Input Token → Embedding → 28 × Transformer Block
          → Final RMSNorm → LM Head → Output Logits
```

先把 Block 当作黑盒：它接收 hidden states，处理后交给下一阶段。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。

## 2. 极简模型信息卡

这些数字只用来帮助我们读懂接下来的模块树，本实验不展开参数组成。

In [ ]:
config = model.config
parameter_count = sum(parameter.numel() for parameter in model.parameters())

print(f"模型: {type(model).__name__}")
print(f"参数量: {parameter_count:,}（约 {parameter_count / 1e6:.1f}M）")
print(f"词表大小: {config.vocab_size:,}")
print(f"隐藏向量维度: {config.hidden_size}")
print(f"Transformer Block 数量: {config.num_hidden_layers}")


## 3. 从真实模型对象生成精简模块树

下面的树不是手写示意图。代码会读取刚刚加载的 `model` 对象，自动取得模块类名、层数和维度。

阅读缩进的方法：

- `model` 和 `lm_head` 是 `Qwen3ForCausalLM` 直接包含的两个顶层部分；
- `embed_tokens`、`layers`、`norm` 都位于内部的 `Qwen3Model` 中；
- `layers × 28` 暂时作为一组黑盒，不在本实验继续展开。

## 4. 把流水线和模块树连起来

现在只做顶层对应，不展开 Block：

| 流水线阶段 | 真实 Python 对象 | 这一阶段做什么 |
|---|---|---|
| Input Token | `input_ids`（稍后传给模型） | 输入 token 的整数编号 |
| Embedding | `model.model.embed_tokens` | 把编号查成 1024 维 hidden states |
| 28 × Transformer Block | `model.model.layers` | 连续经过 28 个黑盒 Block |
| Final RMSNorm | `model.model.norm` | 对最后的 hidden states 做归一化 |
| LM Head | `model.lm_head` | 从 1024 维映射到整个词表 |
| Output Logits | `model.lm_head` 的输出 | 每个候选下一个 token 的原始分数 |

这里要区分两种关系：

- **流水线箭头**表示数据先后经过模块；
- **模块树缩进**表示 Python 对象的包含关系。

In [ ]:
backbone = model.model
embedding = backbone.embed_tokens
blocks = backbone.layers
final_norm = backbone.norm
lm_head = model.lm_head

print(type(model).__name__)
print(f"├── model: {type(backbone).__name__}")
print(
    f"│   ├── embed_tokens: {type(embedding).__name__}"
    f"({embedding.num_embeddings:,}, {embedding.embedding_dim})"
)
print(
    f"│   ├── layers: {type(blocks).__name__} × {len(blocks)} "
    f"[{type(blocks[0]).__name__}]"
)
print(f"│   └── norm: {type(final_norm).__name__}({config.hidden_size})")
print(
    f"└── lm_head: {type(lm_head).__name__}"
    f"({lm_head.in_features}, {lm_head.out_features}, bias={lm_head.bias is not None})"
)


## 关键权重矩阵

PyTorch 的线性层权重形状写作 `(输出维度, 输入维度)`。例如 Q 投影的 `(2048, 1024)` 表示：每个 1024 维输入向量会被变换为 2048 维 Q 向量。

Qwen3-0.6B 使用 GQA（Grouped-Query Attention）：16 个 Query 头，但只有 8 个 Key 头和 8 个 Value 头。因为每头 128 维，所以 Q 是 `16 × 128 = 2048` 维，而 K/V 各是 `8 × 128 = 1024` 维。


In [ ]:
parameters = (
    ('token embedding', model.model.embed_tokens.weight),
    ('Q projection', layer.self_attn.q_proj.weight),
    ('K projection', layer.self_attn.k_proj.weight),
    ('V projection', layer.self_attn.v_proj.weight),
    ('attention output projection', layer.self_attn.o_proj.weight),
    ('MLP gate projection', layer.mlp.gate_proj.weight),
    ('MLP up projection', layer.mlp.up_proj.weight),
    ('MLP down projection', layer.mlp.down_proj.weight),
    ('language-model head', model.lm_head.weight),
)

for name, parameter in parameters:
    print(f'{name:28} {tuple(parameter.shape)}')

shared_weights = (
    model.model.embed_tokens.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)
print(f'\n输入 embedding 与 lm_head 是否共享同一块权重: {shared_weights}')


## 小结与思考

你已经从真实权重中确认：Qwen3-0.6B 有 28 个重复 block，内部主通道宽度是 1024，attention 使用 16 个 Q 头与 8 个 KV 头，MLP 会把 1024 维暂时扩展到 3072 维再压回 1024 维。

试着回答这三个问题：

1. 为什么 28 个 block 都保持输入和输出为 1024 维，而不是每层都继续变宽？
2. 为什么 Q 是 2048 维，但 K 和 V 各只有 1024 维？
3. 为什么模型可以让输入 embedding 和输出 lm_head 共享同一块词表矩阵？

下一实验会亲手观察一个 token ID 如何从 `embed_tokens` 变成 1024 维向量。
